# Climate experiments with a radiation code

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/PySoc/blob/main/notebooks/02_perturbation_experiments.ipynb)

In [Notebook 1](https://colab.research.google.com/github/evanwellmeyer/PySoc/blob/main/notebooks/01_how_radiation_works.ipynb)
we looked inside SOCRATES, the radiation scheme of the Isca climate model. Here we use it as a
laboratory. Each activity **perturbs** the atmosphere (more CO₂, a warmer surface, a cloud) and
measures the response. Together they build up to an estimate of the planet's **climate sensitivity**:
how much the surface warms when CO₂ doubles.

| Activity | Question |
|---|---|
| 1 | How much does doubling CO₂ change the energy balance? |
| 2 | Why does each doubling of CO₂ have about the same effect? |
| 3 | How can we tell a CO₂ warming from a warming by the Sun? |
| 4 | How strongly does a warmer planet push back, and how does water vapour change that? |
| 5 | Which layers of the atmosphere matter most? (using gradients) |
| 6 | When do clouds warm the planet and when do they cool it? |
| 7 | How much does melting ice change the absorbed sunlight? |
| 8 | *Challenge:* build a radiative–convective equilibrium model and measure climate sensitivity |

**How to work:** run the cells in order (later activities use earlier results). Before each experiment,
write down a **prediction** in the box provided, then compare. Hints are hidden under *Hint*: try
without them first.

In [ ]:
# @title Setup: run this cell first (it takes about a minute on Colab)
import importlib, os, subprocess, sys, time

if os.path.isdir("../pysoc") and os.path.abspath("..") not in sys.path:
    sys.path.insert(0, os.path.abspath(".."))  # running inside a copy of the repository
try:
    import pysoc
except ImportError:  # on Colab: install PySoc from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/evanwellmeyer/PySoc"],
                   check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import torch
from cycler import cycler

from pysoc.column import (make_column, liquid_cloud, hydrostatic_heights, saturation_specific_humidity,
                          specific_humidity)
from pysoc.isca import IscaSocrates, GAS_NAMES, CP_AIR, GRAV, RDGAS
from pysoc.spectra import ga7_spectral_files

torch.set_flush_denormal(True)  # avoids a slowdown in float64 on CPUs
torch.set_num_threads(min(4, torch.get_num_threads()))  # one column runs fastest on a few threads
lw_file, sw_file = ga7_spectral_files()  # downloads the SOCRATES GA7 spectral files once
model = IscaSocrates(lw_file, sw_file)

BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LW_COLOR, SW_COLOR, NET_COLOR = BLUE, ORANGE, INK
plt.rcParams.update({
    "figure.dpi": 100, "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 10, "font.size": 10,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "lines.linewidth": 2, "legend.frameon": False,
    "axes.prop_cycle": cycler(color=[BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED]),
})


def radiation(col, albedo=0.3, insolation=340.0, coszen=0.5, cloud=None, remove=()):
    """Run SOCRATES on a column (see Notebook 1). Returns a dict of outputs named as in Isca."""
    ids = {name: i for i, name in GAS_NAMES.items()}
    model.config.exclude_gases = frozenset(ids[name] for name in remove)
    try:
        rrsun = insolation / (coszen * model.config.stellar_constant)
        return model(**col, albedo=albedo, coszen=coszen, rrsun=rrsun, delta_t=0.0, **(cloud or {}))
    finally:
        model.config.exclude_gases = frozenset()


def toa_net(out):
    """Net energy into the planet at the top of the atmosphere: absorbed sunlight minus OLR (W/m2)."""
    return out["soc_toa_sw"] - out["soc_olr"]


def warmed(col, dT=1.0, fixed_rh=False):
    """A copy of the column with the air and surface warmer by dT (K).

    dT is a number, or one value per layer (then the surface warms like the lowest layer).
    fixed_rh=True raises the water vapour so the relative humidity of every layer stays the same.
    """
    new = dict(col)
    dT = torch.broadcast_to(torch.as_tensor(dT, dtype=col["temp"].dtype), col["temp"].shape)
    new["temp"] = col["temp"] + dT
    new["t_surf"] = col["t_surf"] + dT[..., -1]
    new["z_full"], new["z_half"] = hydrostatic_heights(new["temp"], col["p_half"], col["p_full"])
    if fixed_rh:
        new["q"] = col["q"] * saturation_specific_humidity(new["temp"], col["p_full"]) \
            / saturation_specific_humidity(col["temp"], col["p_full"])
    return new


def pressure_axis(ax, top=0.01):
    ax.set_yscale("log")
    ax.set_ylim(1000, top)
    ax.set_yticks([t for t in (1000, 300, 100, 30, 10, 3, 1, 0.3, 0.1, 0.03, 0.01) if t >= top])
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    ax.set_ylabel("pressure (hPa)")


def band_label(sp, b):
    lo, hi = sp.wavelength_short[b] * 1e6, sp.wavelength_long[b] * 1e6
    return f"{b + 1}: {lo:.3g}–{hi:.3g} µm" if hi < 1000 else f"{b + 1}: {lo:.3g}–{hi:.0f} µm"


hpa = lambda p: (p / 100).detach().numpy()  # noqa: E731  Pa -> hPa
per_day = lambda rate: (rate * 86400).detach().numpy()  # noqa: E731  K/s -> K/day
lw_spec = model.lw.spectrum.sp
print("Ready.")

## The control climate

All experiments start from the same **control** column: a pre-industrial atmosphere (280 ppm CO₂)
over a surface at 288 K, with global-mean sunlight (340 W/m²) and no clouds. We use a pressure of
200 hPa as a simple stand-in for the tropopause.

In [ ]:
control = make_column(t_surf=288.0, co2_ppmv=280.0)
o_ctl = radiation(control)
p = hpa(control["p_full"])
p_half = hpa(control["p_half"])
p_half[0] = 0.01  # the top half level is at p = 0; plot it at the top of the axis
TROP = int(np.argmin(np.abs(p_half - 200)))  # half level closest to 200 hPa
print(f"OLR {float(o_ctl['soc_olr']):.1f} W/m²,  absorbed sunlight {float(o_ctl['soc_toa_sw']):.1f} W/m²,  "
      f"tropopause level {p_half[TROP]:.0f} hPa")

---
## Activity 1: Doubling CO₂

CO₂ has risen from 280 ppm before the industrial revolution to over 420 ppm today. Let's double it to
560 ppm, keeping everything else (temperatures, humidity) fixed. The change in the energy balance
is called the **radiative forcing**.

✏️ **Predict:** will the OLR go up or down? By a few W/m², or tens of W/m²?

*Your prediction:*

In [ ]:
doubled = make_column(t_surf=288.0, co2_ppmv=560.0)
o_2x = radiation(doubled)

print(f"change in OLR:                    {float(o_2x['soc_olr'] - o_ctl['soc_olr']):+.2f} W/m²")
print(f"change in absorbed sunlight:      {float(o_2x['soc_toa_sw'] - o_ctl['soc_toa_sw']):+.2f} W/m²")
print(f"forcing at the top of atmosphere: {float(toa_net(o_2x) - toa_net(o_ctl)):+.2f} W/m²")

# net downward flux (LW + SW) at every level: its change is the forcing at that level
net_down = lambda o: -(o["soc_flux_lw"] + o["soc_flux_sw"])  # noqa: E731
forcing_profile = (net_down(o_2x) - net_down(o_ctl)).numpy()
F_2x = forcing_profile[TROP]
print(f"forcing at the tropopause:        {F_2x:+.2f} W/m²")

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.3), gridspec_kw=dict(width_ratios=[1, 1.4]))
ax.plot(forcing_profile, p_half, color=INK)
ax.axhline(p_half[TROP], color=MUTED, linewidth=1)
ax.annotate("tropopause", (0.02, p_half[TROP]), xytext=(0, -12), textcoords="offset points",
            xycoords=("axes fraction", "data"), color=INK2, fontsize=9)
pressure_axis(ax)
ax.set_xlabel("extra energy flowing down (W/m²)")
ax.set_title("Forcing at each level")
d_band = (o_2x["soc_spectral_olr"] - o_ctl["soc_spectral_olr"]).numpy()
y = np.arange(len(d_band))
ax2.barh(y, d_band, height=0.7, color=LW_COLOR)
ax2.axvline(0, color=AXIS, linewidth=1)
ax2.set_yticks(y, [band_label(lw_spec, b) for b in y])
ax2.invert_yaxis()
ax2.grid(axis="y", visible=False)
ax2.set_xlabel("change in OLR (W/m²)")
ax2.set_title("Change in OLR, band by band")
fig.tight_layout()
plt.show()

**Questions**
1. Which band does the extra CO₂ affect most, and why? Band 5 is the "window": why does it change at all?
   (Hint: CO₂ has two weak bands near 9.4 and 10.4 µm.)
2. The forcing at the tropopause is almost twice the forcing at the top of the atmosphere. Where does
   the difference go?
3. The IPCC's best estimate for the forcing from doubling CO₂ is about 3.9 W/m². It lets the
   stratosphere adjust and includes clouds, so it lies between our two numbers. Why would clouds make
   the forcing *smaller*?

<details><summary>Hint</summary>

For question 2: look at the forcing profile above the tropopause. More CO₂ in the stratosphere means
more emission from the stratosphere, both upward and downward. For question 3: a cloud top emits
like a cold surface. Above a high cloud, adding CO₂ makes less difference.
</details>

---
## Activity 2: The logarithmic effect of CO₂

PySoc can run many columns at once. Here we build eight columns with CO₂ from 35 to 4480 ppm (each
double the last) and run them in **one** call.

✏️ **Predict:** is the forcing from 280→560 ppm bigger, smaller or the same as from 2240→4480 ppm?

*Your prediction:*

In [ ]:
co2_values = torch.tensor([35.0, 70.0, 140.0, 280.0, 560.0, 1120.0, 2240.0, 4480.0])
many = make_column(t_surf=288.0, co2_ppmv=co2_values)  # eight columns at once
o_many = radiation(many)
forcing = (net_down(o_many)[:, TROP] - net_down(o_ctl)[TROP]).numpy()

for c, f in zip(co2_values.tolist(), forcing):
    print(f"{c:7.0f} ppm   forcing {f:+6.2f} W/m²")
print("forcing per doubling:", np.round(np.diff(forcing), 2), "W/m²")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.axhline(0, color=AXIS, linewidth=1)
ax.plot(co2_values.numpy(), forcing, marker="o", markersize=7, color=INK, label="PySoc, clear sky, tropopause")
ax.plot(co2_values.numpy(), 5.35 * np.log(co2_values.numpy() / 280.0), color=MUTED, linewidth=1.5,
        label="5.35 ln(C/280), Myhre et al. (1998)")
ax.set_xscale("log", base=2)
ax.set_xticks(co2_values.tolist(), [f"{c:.0f}" for c in co2_values.tolist()])
ax.set_xlabel("CO₂ (ppm, log scale)")
ax.set_ylabel("forcing (W/m²)")
ax.set_title("Each doubling adds about the same forcing")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

**Questions**
1. Is the forcing per doubling constant? How close is it to a straight line on the log axis?
2. Use your results to estimate the forcing from the increase so far (280 → 420 ppm).
3. Why doesn't the forcing grow in proportion to the amount of CO₂? Use Activity 1's band-by-band plot
   to explain.
4. The simple formula (from calculations with clouds and a global mix of atmospheres) gives smaller
   values than our clear-sky column. Is the *shape* the same?

<details><summary>Hint</summary>

For question 3: at the centre of the 15 µm band the atmosphere is already opaque; adding CO₂ there
only moves the emission level up a little. Most of the change comes from the band's edges, where the
absorption is weaker, and the width of the "opaque" region grows with the logarithm of the amount.
</details>

---
## Activity 3: A fingerprint of CO₂

The Sun's output also varies. Could a brighter Sun explain global warming instead of CO₂? Radiation
gives us a way to tell. Compare how each one changes the **heating rate** at every level:
* doubling CO₂;
* making the Sun 1% brighter (340 → 343.4 W/m²).

✏️ **Predict:** does each one warm or cool the stratosphere?

*Your prediction:*

In [ ]:
o_sun = radiation(control, insolation=340.0 * 1.01)
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
for a, top in ((ax, 0.01), (ax2, 100)):
    a.axvline(0, color=AXIS, linewidth=1)
    a.plot(per_day(o_2x["tdt_rad"] - o_ctl["tdt_rad"]), p, color=INK, label="doubled CO₂")
    a.plot(per_day(o_sun["tdt_rad"] - o_ctl["tdt_rad"]), p, color=SW_COLOR, label="Sun 1% brighter")
    a.set_xlabel("change in heating rate (K/day)")
ax.set_ylim(1000, 0.01)
pressure_axis(ax)
ax.set_title("Whole atmosphere")
ax.legend(loc="lower left")
ax2.set_xlim(-0.1, 0.12)
ax2.set_title("Zoom in on small values")
fig.tight_layout()
plt.show()

**Questions**
1. How do the two patterns differ in the stratosphere? In the troposphere?
2. Weather balloons and satellites show the troposphere warming and the stratosphere *cooling* since the
   1970s. Which cause does that fit?
3. Why does more CO₂ cool the stratosphere? (Hint: in the stratosphere CO₂ both absorbs radiation from
   below and emits its own. Which process gains more when CO₂ increases?)

---
## Activity 4: Feedbacks and climate sensitivity

A forcing makes the planet gain energy, so it warms. A warmer planet radiates more, which pushes back.
The **feedback parameter** λ measures how much more energy escapes per degree of warming (W/m² per K).
The equilibrium warming is then roughly

$$\Delta T \approx \frac{F}{\lambda}.$$

We measure λ by warming the whole column (air and surface) by 1 K:
* **Planck response**: the water vapour content stays fixed;
* **with water-vapour feedback**: the relative humidity stays fixed, so a warmer atmosphere holds more
  water vapour (about 7% more per K), as in the real atmosphere.

✏️ **Predict:** will the water-vapour feedback make λ bigger or smaller? Will the warming be bigger
or smaller?

*Your prediction:*

In [ ]:
o_planck = radiation(warmed(control, 1.0, fixed_rh=False))
o_wv = radiation(warmed(control, 1.0, fixed_rh=True))
lam_planck = float(toa_net(o_ctl) - toa_net(o_planck))
lam_wv = float(toa_net(o_ctl) - toa_net(o_wv))

print(f"Planck response:          lambda = {lam_planck:.2f} W/m² per K")
print(f"with water-vapour feedback: lambda = {lam_wv:.2f} W/m² per K")
print()
print(f"forcing from doubled CO2 (Activity 1, tropopause): {F_2x:.2f} W/m²")
print(f"warming, Planck response only:        {F_2x / lam_planck:.2f} K")
print(f"warming, with water-vapour feedback:  {F_2x / lam_wv:.2f} K")

**Questions**
1. By what factor does the water-vapour feedback amplify the warming?
2. A blackbody at the planet's effective emission temperature (255 K) has
   $\lambda = 4\sigma T^3 \approx 3.8$ W/m² per K. How does the Planck response compare?
3. The IPCC's best estimate of the warming from doubled CO₂ is 3 K (likely range 2.5–4 K). Which
   feedbacks does our column leave out?
4. *Explore:* in the real tropics, the upper troposphere warms more than the surface (the **lapse-rate
   feedback**). The cell below warms the surface by about 1 K, the air just below 200 hPa by 2 K, and the
   stratosphere by 1 K as before. Does λ go up or down? Why does fixing the relative humidity cancel part of
   the change? Try other profiles.

<details><summary>Hint</summary>

For question 3: think about clouds, sea ice and snow, and how the temperature profile changes.
</details>

In [ ]:
p_full = control["p_full"]
extra = torch.where(p_full > 200e2, (1e5 - p_full) / (1e5 - 200e2), torch.zeros_like(p_full))  # 0 at the surface, 1 at 200 hPa
dT = 1.0 + extra
print("warming profile (K), top to bottom:", np.round(dT.numpy(), 2))
for fixed_rh in (False, True):
    o_lapse = radiation(warmed(control, dT, fixed_rh=fixed_rh))
    lam = float(toa_net(o_ctl) - toa_net(o_lapse)) / float(dT[-1])
    print(f"fixed relative humidity: {fixed_rh!s:5}  lambda = {lam:.2f} W/m² per K of surface warming")

---
## Activity 5: Which layers matter? Asking the model with gradients

PySoc is written in PyTorch, so it can compute **gradients**: how much an output changes when you
nudge each input. One backward pass gives the sensitivity of the OLR to the temperature and humidity
of *every* layer at once. Climate scientists call these profiles **radiative kernels**. (The
original Fortran cannot do this: you would have to perturb each layer one at a time.)

✏️ **Predict:** is the OLR most sensitive to the temperature near the surface, in the upper
troposphere, or in the stratosphere?

*Your prediction:*

In [ ]:
temp = control["temp"].clone().requires_grad_(True)
t_surf = control["t_surf"].clone().requires_grad_(True)
log_q = torch.log(control["q"]).clone().requires_grad_(True)
o = radiation(dict(control, temp=temp, t_surf=t_surf, q=torch.exp(log_q)))
dOLR_dT, dOLR_dTs, dOLR_dlogq = torch.autograd.grad(o["soc_olr"], [temp, t_surf, log_q])

dp = torch.diff(control["p_half"]) / 100  # layer thickness in hPa
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 4.3), sharey=True)
ax.axvline(0, color=AXIS, linewidth=1)
ax.plot((dOLR_dT / dp * 100).numpy(), p, color=RED)
ax.set_xlabel("W/m² per K, per 100 hPa of air")
ax.set_title("OLR sensitivity to temperature")
ax2.axvline(0, color=AXIS, linewidth=1)
ax2.plot((dOLR_dlogq / dp * 100 * np.log(1.1)).numpy(), p, color=BLUE)
ax2.set_xlabel("W/m² for 10% more water vapour, per 100 hPa")
ax2.set_title("OLR sensitivity to water vapour")
pressure_axis(ax)
fig.tight_layout()
plt.show()
print(f"sum over layers of dOLR/dT: {float(dOLR_dT.sum()):.2f},  dOLR/dT_surface: {float(dOLR_dTs):.2f} W/m² per K")
above = control["p_full"] < 200e2
print(f"share of the air's total temperature sensitivity from above 200 hPa: "
      f"{100 * float(dOLR_dT[above].sum() / dOLR_dT.sum()):.0f}% (that air is {100 * float(dp[above].sum()) / 1000:.0f}% of the mass)")

**Check the gradient:** warm a single layer by 1 K and compare the change in OLR with the gradient.
Change `layer` and rerun. (They agree closely but not exactly. Why? Try a 0.1 K warming instead, and
compare with 0.1 × the gradient.)

In [ ]:
layer = 25  # 0 is the top layer, 39 the bottom one
bump = torch.zeros_like(control["temp"])
bump[layer] = 1.0
single = dict(control, temp=control["temp"] + bump)
print(f"layer {layer} at {p[layer]:.0f} hPa")
print(f"  OLR change from rerunning the model: {float(radiation(single)['soc_olr'] - o_ctl['soc_olr']):.5f} W/m²")
print(f"  OLR change predicted by the gradient: {float(dOLR_dT[layer]):.5f} W/m²")

**Questions**
1. Add up the temperature sensitivities (layers plus surface). Compare with the Planck response from
   Activity 4. Should they agree?
2. Per 100 hPa, the OLR is most sensitive to the temperature of the upper stratosphere. Yet the
   stratosphere contributes only a small share of the total. How can both be true?
3. A 10% increase in water vapour has a similar effect throughout most of the troposphere. But the upper
   troposphere holds about 100 times less water than the air near the surface. Per *gram* of added
   water, which region matters more? Why does this make upper-tropospheric humidity so important?
4. Extra water vapour between about 1 and 30 hPa *increases* the OLR. Why? (Hint: compare the
   stratosphere's temperature with the tropopause's.)

---
## Activity 6: Clouds, high and low

Isca's simple cloud scheme passes SOCRATES liquid clouds. Here we move a cloud layer 100 hPa thick
through the troposphere and record its **cloud radiative effect** (CRE) at the top of the atmosphere:
positive means the cloud warms the planet. (Real high clouds are made of ice; Isca's scheme treats all
clouds as liquid water.)

✏️ **Predict:** which clouds warm the planet, high or low? Does the answer depend on thickness?

*Your prediction:*

In [ ]:
def cloud_effect(cloud_top_hPa, lwp, thickness_hPa=100.0):
    cloud = liquid_cloud(control, cloud_top_hPa * 100.0, (cloud_top_hPa + thickness_hPa) * 100.0, lwp=lwp)
    o = radiation(control, cloud=cloud)
    lw = float(o["soc_olr_clr"] - o["soc_olr"])
    sw = float(o["soc_toa_sw"] - o["soc_toa_sw_clr"])
    return lw, sw, lw + sw


tops = np.arange(150, 900, 50)
lwp_values = np.array([2, 5, 10, 20, 50, 100, 200, 400])
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.3))
by_height = np.array([cloud_effect(t, lwp=50) for t in tops])
ax.axvline(0, color=AXIS, linewidth=1)
for i, (name, color) in enumerate((("longwave", LW_COLOR), ("shortwave", SW_COLOR), ("net", NET_COLOR))):
    ax.plot(by_height[:, i], tops, marker="o", markersize=5, color=color, label=name)
ax.set_ylim(900, 100)
ax.set_ylabel("cloud-top pressure (hPa)")
ax.set_xlabel("cloud radiative effect (W/m²)")
ax.set_title("Moving a cloud (50 g/m²) up and down")
ax.legend(loc="lower right")
for top, color, name in ((250, BLUE, "high cloud (top 250 hPa)"), (800, AQUA, "low cloud (top 800 hPa)")):
    net = [cloud_effect(top, lwp)[2] for lwp in lwp_values]
    ax2.plot(lwp_values, net, marker="o", markersize=5, color=color, label=name)
ax2.axhline(0, color=AXIS, linewidth=1)
ax2.set_xscale("log")
ax2.set_xticks(lwp_values.tolist(), [str(v) for v in lwp_values])
ax2.set_xlabel("liquid water path (g/m², log scale)")
ax2.set_ylabel("net cloud radiative effect (W/m²)")
ax2.set_title("Thin and thick clouds")
ax2.legend(loc="lower left")
fig.tight_layout()
plt.show()

**Questions**
1. Why does the longwave effect grow with cloud height while the shortwave effect hardly changes?
2. Which kind of cloud warms the planet overall? Which cools it?
3. Global warming may change how many low clouds there are, and how high the high clouds reach. Using
   your plots, explain why clouds are the biggest source of uncertainty in climate sensitivity.

---
## Activity 7: Ice, snow and ocean

Sea ice and snow reflect most sunlight (albedo about 0.6–0.8); open ocean absorbs most of it (albedo
about 0.06). Here we change the surface albedo under the clear-sky column.

✏️ **Predict:** if the surface albedo drops from 0.6 to 0.06, the surface reflects 54% less of the
sunlight reaching it. Will the planet absorb 54% × 340 ≈ 184 W/m² more? More, or less?

*Your prediction:*

In [ ]:
albedos = torch.tensor([0.06, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8])
many_same = make_column(t_surf=torch.full((len(albedos),), 288.0), co2_ppmv=280.0)  # nine identical columns
o_alb = radiation(many_same, albedo=albedos)
absorbed = o_alb["soc_toa_sw"].numpy()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(albedos.numpy(), absorbed, marker="o", markersize=6, color=SW_COLOR)
ax.set_xlabel("surface albedo")
ax.set_ylabel("sunlight absorbed by the planet (W/m²)")
ax.set_title("Brighter surfaces, less absorbed sunlight")
fig.tight_layout()
plt.show()
gain = absorbed[0] - absorbed[6]
print(f"albedo 0.6 -> 0.06: the planet absorbs {gain:.0f} W/m² more (the simple estimate was 184 W/m²)")
print(f"with lambda = {lam_wv:.2f} W/m² per K from Activity 4, that would be worth {gain / lam_wv:.0f} K of local warming")

**Questions**
1. Why is the extra absorbed sunlight less than the simple estimate? (Hint: what happens to sunlight
   before it reaches the surface, and after it is reflected?)
2. Explain why melting ice is a *positive* feedback.
3. The local warming you computed is far larger than what is actually observed when sea ice melts. What
   limits it? (Think about the seasons, clouds and the ocean moving heat around.)

---
## Activity 8 (challenge): Radiative–convective equilibrium

In 1967, Syukuro Manabe and Richard Wetherald built a one-column model of the atmosphere: radiation
plus a simple rule for convection. They used it to make the first reliable estimate of the warming from
doubled CO₂. Manabe shared the 2021 Nobel Prize in Physics for this line of work. We now have all the
pieces to repeat their experiment with a modern radiation code.

The model steps forward in time, one day per step:
1. radiation heats or cools each layer (`tdt_rad`) and the surface (its net radiation);
2. **convective adjustment**: if the air above the surface is too cold (the temperature falls with height
   faster than a critical lapse rate, 6.5 K/km), convection mixes it with the surface into one region with
   exactly that lapse rate, keeping the region's total energy. The region grows upward until the air
   above it is stable;
3. the water vapour is recomputed for a fixed relative humidity (in parts B and C).

The surface is a 1 m deep layer of water. Read the two functions below, then run them.

In [ ]:
C_SURF = 4.2e6  # heat capacity of 1 m of water, J/m2 per K


def convective_adjustment(temp, t_surf, p_full, p_half, lapse_rate=6.5):
    """Mix the surface and the unstable air above it to the critical lapse rate (K/km), conserving energy."""
    L = temp.shape[-1]
    # work on numpy arrays of shape (columns, L + 1); the surface is the last "layer"
    T = np.concatenate([temp.numpy(), t_surf.numpy()[..., None]], -1).reshape(-1, L + 1)
    pres = np.concatenate([p_full.numpy(), p_half.numpy()[..., -1:]], -1).reshape(-1, L + 1)
    heat_capacity = np.concatenate([CP_AIR * np.diff(p_half.numpy(), axis=-1) / GRAV,
                                    np.full(temp.shape[:-1] + (1,), C_SURF)], -1).reshape(-1, L + 1)
    # with a constant lapse rate, T(p) = T_surface * (p / p_surface) ** (R * lapse_rate / g)
    shape = (pres / pres[:, -1:]) ** (RDGAS * lapse_rate * 1e-3 / GRAV)
    for i in range(T.shape[0]):
        top, t_base = L, T[i, L]  # the mixed region is levels top..L; it starts as just the surface
        while top > 0 and T[i, top - 1] < t_base * shape[i, top - 1]:  # the air above is too cold: mix it in
            top -= 1
            c = heat_capacity[i, top:]
            t_base = (c * T[i, top:]).sum() / (c * shape[i, top:]).sum()  # keeps the total energy
        T[i, top:] = t_base * shape[i, top:]
    T = T.reshape(temp.shape[:-1] + (L + 1,))
    return torch.from_numpy(T[..., :-1].copy()), torch.from_numpy(T[..., -1].copy())


def equilibrate(col, days=400, convection=True, fixed_rh=True, albedo=0.23, lapse_rate=6.5, rh_surface=0.8):
    """Step the column forward one day at a time; returns the final column and the net TOA flux each day."""
    col = {k: v.clone() for k, v in col.items()}
    dt = 86400.0
    history = []
    start = time.time()
    with torch.no_grad():
        for day in range(days):
            o = radiation(col, albedo=albedo)
            temp = col["temp"] + o["tdt_rad"] * dt
            t_surf = col["t_surf"] + dt * (o["soc_surf_flux_sw"] - o["soc_surf_flux_lw"]) / C_SURF
            if convection:
                temp, t_surf = convective_adjustment(temp, t_surf, col["p_full"], col["p_half"], lapse_rate)
            if not bool(torch.isfinite(temp).all()):
                raise RuntimeError("the temperatures blew up; try a shorter run or smaller changes")
            col["temp"], col["t_surf"] = temp, t_surf
            col["z_full"], col["z_half"] = hydrostatic_heights(temp, col["p_half"], col["p_full"])
            if fixed_rh:
                col["q"] = specific_humidity(temp, col["p_full"], col["p_half"][..., -1], rh_surface)
            history.append(toa_net(o).numpy().copy())
            if (day + 1) % 100 == 0:
                print(f"day {day + 1}: surface {np.round(t_surf.numpy(), 2)} K, "
                      f"TOA imbalance {np.round(history[-1], 2)} W/m²  ({time.time() - start:.0f} s)")
    return col, np.array(history)

### Part A: radiation alone

Start from an atmosphere at a uniform 250 K and let radiation alone (no convection) find its
equilibrium. The water vapour stays fixed at the control amounts. The surface albedo is 0.23, which
gives a surface temperature close to today's in part C. (Each part takes under a minute.)

✏️ **Predict:** will the equilibrium surface be warmer or colder than 288 K?

*Your prediction:*

In [ ]:
isothermal = dict(control, temp=torch.full_like(control["temp"], 250.0), t_surf=torch.tensor(250.0, dtype=torch.float64))
isothermal["z_full"], isothermal["z_half"] = hydrostatic_heights(isothermal["temp"], control["p_half"], control["p_full"])
radiative, hist_a = equilibrate(isothermal, days=500, convection=False, fixed_rh=False)

### Part B: radiation plus convection

The same experiment, now with convective adjustment.

In [ ]:
rce, hist_b = equilibrate(isothermal, days=300, convection=True, fixed_rh=False)

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10.5, 4.5), gridspec_kw=dict(width_ratios=[1.3, 1]))
for c, color, name in ((radiative, RED, "radiation only"), (rce, INK, "radiation + convection")):
    ax.plot(c["temp"].numpy(), p, color=color, label=name)
    ax.plot(float(c["t_surf"]), 1000, marker="s", markersize=8, color=color, clip_on=False)
ax.plot(control["temp"].numpy(), p, color=MUTED, linewidth=1.5, label="control column")
pressure_axis(ax)
ax.set_xlabel("temperature (K); squares = surface")
ax.set_title("Equilibrium temperature profiles")
ax.legend(loc="upper right")
ax2.axhline(0, color=AXIS, linewidth=1)
ax2.plot(hist_a, color=RED, label="radiation only")
ax2.plot(hist_b, color=INK, label="radiation + convection")
ax2.set_ylim(-20, 40)
ax2.set_xlabel("day")
ax2.set_ylabel("net energy in at the top (W/m²)")
ax2.set_title("Approach to equilibrium")
ax2.legend(loc="upper right")
fig.tight_layout()
plt.show()
print(f"surface temperature: radiation only {float(radiative['t_surf']):.1f} K, "
      f"with convection {float(rce['t_surf']):.1f} K")

**Questions**
1. Radiation alone makes the surface much warmer than the air just above it, and the lower atmosphere
   very steep. Why would such an atmosphere not survive in reality?
2. Both equilibria have a stratosphere whose temperature rises with height. What causes it?
3. Where does convection move energy from, and to?

### Part C: climate sensitivity

Now the real experiment: two columns, one at 280 ppm and one at 560 ppm, both with fixed relative
humidity, run to equilibrium side by side in one batch.

✏️ **Predict:** using Activity 4, how much warmer will the 560 ppm column's surface be?

*Your prediction:*

In [ ]:
pair = make_column(t_surf=288.0, co2_ppmv=torch.tensor([280.0, 560.0]))
equilibrium, hist_c = equilibrate(pair, days=400, convection=True, fixed_rh=True)
ecs = float(equilibrium["t_surf"][1] - equilibrium["t_surf"][0])
print(f"\nsurface temperature at 280 ppm: {float(equilibrium['t_surf'][0]):.2f} K")
print(f"surface temperature at 560 ppm: {float(equilibrium['t_surf'][1]):.2f} K")
print(f"warming from doubled CO2 (equilibrium climate sensitivity of this model): {ecs:.2f} K")

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.axvline(0, color=AXIS, linewidth=1)
ax.plot((equilibrium["temp"][1] - equilibrium["temp"][0]).numpy(), p, color=INK)
ax.plot(ecs, 1000, marker="s", markersize=8, color=INK, clip_on=False)
pressure_axis(ax)
ax.set_xlabel("temperature change (K); square = surface")
ax.set_title("Equilibrium response to doubled CO₂")
fig.tight_layout()
plt.show()

**Questions**
1. Compare the warming with your estimate from Activity 4 and with Manabe and Wetherald's 2.4 K (1967,
   fixed relative humidity). Why might they differ?
2. The stratosphere cools in equilibrium too. Compare with Activity 3.
3. Rerun part C with `lapse_rate=9.8` (a dry atmosphere) or with `rh_surface=0.5`. Which way does the
   sensitivity move, and why? (Pass the new values to `equilibrate`.)
4. List the feedbacks this model leaves out. For each, do you expect it to raise or lower the warming?

---
## Explore on your own

A few more experiments to try, starting from the code above:
* **Ozone hole:** `make_column(ozone_du=150)`. What happens to the stratospheric heating?
* **Methane:** double it with `model.config.well_mixed[6] *= 2` (gas 6 is CH₄; set it back afterwards).
  How does its forcing compare with CO₂'s, per molecule?
* **Faint young Sun:** 4 billion years ago the Sun was about 25% dimmer. How much CO₂ would the early
  Earth have needed to keep the same OLR balance? (Use `insolation=0.75 * 340` and a sweep of CO₂.)
* **A different planet:** Isca is used for exoplanets too. Try a hotter, wetter column
  (`t_surf=310`, `rh_surface=0.9`) and repeat Activity 4. Does the water-vapour feedback get stronger?